# OBD Soft-Prompt Training (OpenTSLM-SP)

Soft-prompt alignment using OpenTSLMSP with OBD-CoT dataset.

## 1) Setup

In [ ]:
from pathlib import Path
import sys

CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'pyproject.toml').exists() else CWD.parent

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "open_flamingo"))
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_
from transformers import get_linear_schedule_with_warmup

from opentslm.model.llm.OpenTSLMSP import OpenTSLMSP

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = 'cuda'
# elif torch.backends.mps.is_available():
#     DEVICE = 'mps'
else:
    DEVICE = 'cpu'

DATA_PATH = PROJECT_ROOT / 'data/obd_cot_gpt5.jsonl'
LLM_ID = 'google/gemma-3-270m'
# LLM_ID = "meta-llama/Llama-3.2-1B"
BATCH_SIZE = 1
NUM_EPOCHS = 200
LR = 2e-4
WARMUP_FRAC = 0.1
GRAD_CLIP = 1.0
LLM_DTYPE = torch.bfloat16  # MPS-safe
patience = 5


## 2) Load dataset

In [7]:
rows = [json.loads(x) for x in DATA_PATH.read_text(encoding='utf-8').splitlines() if x.strip()]
print(f'Amostras: {len(rows)}')
rows[0]

# Filter out 'congested' class
rows = [r for r in rows if r.get('label') != 'congested']
print('Amostras (sem congested):', len(rows))

# Rebuild pre_prompt to remove label leakage (keep two options, but no 'correct' hint)
_NO_LEAK_PROMPT = """You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[{label}]
[{dis}]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST end your response with \"Answer:\"."""

def _build_no_leak_prompt(r):
    label = r.get('label') or ''
    dis = r.get('dissimilar_label') or ''
    return _NO_LEAK_PROMPT.format(label=label, dis=dis)

for r in rows:
    r['pre_prompt'] = _build_no_leak_prompt(r)

# sanity check
print('pre_prompt sample (no leak):')
print(rows[0]['pre_prompt'])


Amostras: 195
Amostras (sem congested): 193
pre_prompt sample (no leak):
You are shown a time-series plot of vehicle telemetry over a 120-sample window.
This data corresponds to one of two possible activities:
[economical]
[aggressive]
Your task is to classify the activity based on analysis of the data.
Instructions:
- Begin by analyzing the time-series without assuming a specific label.
- Think step-by-step about what the observed patterns suggest regarding movement intensity and behavior.
- Write your rationale as a single, natural paragraph, do not use bullet points, numbered steps, or section headings.
- Do not refer back to the plot or to the act of visual analysis in your rationale; the plot is only for reference but you should reason about the time-series data.
- Do not assume any answer at the beginning, analyze as if you do not yet know which class is correct.
- Do not mention either class label until the final sentence.
- Make sure that your last word is the answer. You MUST 

## 3) Train/val/test split

In [8]:
idx = list(range(len(rows)))
random.shuffle(idx)
n = len(idx)
train_end = max(1, int(0.6*n))
val_end = max(train_end+1, int(0.8*n)) if n > 2 else train_end

train_rows = [rows[i] for i in idx[:train_end]]
val_rows = [rows[i] for i in idx[train_end:val_end]]
test_rows = [rows[i] for i in idx[val_end:]]

if len(val_rows) == 0 and len(train_rows) > 1:
    val_rows = [train_rows.pop()]
if len(test_rows) == 0 and len(train_rows) > 1:
    test_rows = [train_rows.pop()]

print('train/val/test:', len(train_rows), len(val_rows), len(test_rows))


train/val/test: 115 39 39


## 4) Dataset wrapper for OpenTSLM-SP

In [9]:
FEATURE_LABELS = [
    'Speed',
    'RPM',
    'Engine Load',
    'Throttle Position',
]

def summarize_series(label, values, n_points=8):
    values = values.detach().cpu().float()
    n = values.numel()
    idx = torch.linspace(0, max(n - 1, 0), steps=min(n_points, n)).long()
    samples = ', '.join(f'{values[i].item():.1f}' for i in idx)
    delta = values[-1].item() - values[0].item()
    if delta > 1e-3:
        trend = 'overall increasing'
    elif delta < -1e-3:
        trend = 'overall decreasing'
    else:
        trend = 'roughly flat'
    return (
        f'{label} series with mean {values.mean().item():.2f}, '
        f'std {values.std(unbiased=False).item():.2f}, '
        f'min {values.min().item():.2f}, max {values.max().item():.2f}, '
        f'trend {trend}, sampled points [{samples}]:'
    )

class OBDSPDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r = self.rows[i]
        ts = torch.tensor(r['time_series'], dtype=torch.float32)  # [F, T]
        # Expect 4 features: speed, rpm, load, throttle
        if ts.ndim != 2 or ts.shape[0] < 4:
            raise ValueError('time_series shape unexpected')
        # Build per-feature series and richer textual summaries.
        time_series = [ts[0], ts[1], ts[2], ts[3]]
        time_series_text = [
            summarize_series(label, series)
            for label, series in zip(FEATURE_LABELS, time_series)
        ]

        return {
            'pre_prompt': r['pre_prompt'],
            'time_series_text': time_series_text,
            'post_prompt': r['post_prompt'],
            'answer': r['answer'],
            'time_series': time_series,
            'id': r['id'],
        }

train_ds = OBDSPDataset(train_rows)
val_ds = OBDSPDataset(val_rows)
test_ds = OBDSPDataset(test_rows)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=lambda x: x)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=lambda x: x)


## 5) Initialize OpenTSLM-SP

In [10]:
model = OpenTSLMSP(llm_id=LLM_ID, device=DEVICE).to(DEVICE)

model.enable_lora(lora_r=16, lora_alpha=32, lora_dropout=0)
# keep fp32 on MPS
model.llm = model.llm.to(dtype=LLM_DTYPE)

trainable = [(n,p) for n,p in model.named_parameters() if p.requires_grad]
print('Trainable params:', sum(p.numel() for _,p in trainable))

optimizer = torch.optim.AdamW([p for _,p in trainable], lr=LR)
total_steps = max(1, NUM_EPOCHS * len(train_loader))
warmup_steps = int(WARMUP_FRAC * total_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)


`torch_dtype` is deprecated! Use `dtype` instead!
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ LoRA enabled:
   LoRA parameters: 3,796,992
   Total trainable parameters: 3,796,992
   Total parameters: 271,895,808
   LoRA %: 1.40%
   Trainable %: 1.40%
Trainable params: 5990784


## 6) Sanity check

In [11]:
batch = next(iter(train_loader))
inputs_embeds, attention_mask = model.pad_and_apply_batch(batch)
print('inputs_embeds', inputs_embeds.shape)
print('attention_mask', attention_mask.shape)


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


inputs_embeds torch.Size([1, 660, 640])
attention_mask torch.Size([1, 660])


## 7) Train + validate

In [12]:
best_val = float('inf')
best_path = 'data/obd_sp_best.pt'
no_improve = 0

for epoch in range(1, NUM_EPOCHS+1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        loss = model.compute_loss(batch)
        if not torch.isfinite(loss):
            print('Loss not finite, aborting epoch.')
            break
        loss.backward()
        clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        train_loss += float(loss.item())
    train_loss /= max(1, len(train_loader))

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in val_loader:
            vloss = model.compute_loss(batch)
            if not torch.isfinite(vloss):
                print('Val loss not finite.')
                break
            val_loss += float(vloss.item())
    val_loss /= max(1, len(val_loader))

    print(f'Epoch {epoch}: train={train_loss:.6f} val={val_loss:.6f}')
    if val_loss < best_val:
        best_val = val_loss
        model.store_to_file(str(best_path))
    else:
        no_improve += 1
        if no_improve >= patience:
            print('Early stopping acionado.')
            break


Epoch 1: train=4.425921 val=3.798396
💾 Saved LoRA adapters with 252 parameters
Epoch 2: train=3.171399 val=2.877722
💾 Saved LoRA adapters with 252 parameters
Epoch 3: train=2.563847 val=2.552123
💾 Saved LoRA adapters with 252 parameters
Epoch 4: train=2.260175 val=2.372419
💾 Saved LoRA adapters with 252 parameters
Epoch 5: train=2.040433 val=2.278784
💾 Saved LoRA adapters with 252 parameters
Epoch 6: train=1.854500 val=2.254187
💾 Saved LoRA adapters with 252 parameters
Epoch 7: train=1.693408 val=2.208738
💾 Saved LoRA adapters with 252 parameters
Epoch 8: train=1.517157 val=2.312339
Epoch 9: train=1.339136 val=2.424670
Epoch 10: train=1.153727 val=2.546518
Epoch 11: train=0.989450 val=2.704649
Epoch 12: train=0.819756 val=3.067587
Early stopping acionado.


## 8) Test inference

In [13]:
# model.load_from_file('data/obd_sp_best.pt')
model.eval()
preds = []
with torch.no_grad():
    for batch in test_loader:
        gen = model.generate(batch, max_new_tokens=256)[0]
        preds.append({
            'id': batch[0]['id'],
            'prediction': gen.strip(),
            'target': batch[0]['answer'],
        })
preds[:3]


[{'id': 'obd_cot_000088',
  'prediction': 'CEROMEOPEOPLE,DISTANCE,EMISSARY,DISTANCE,EMISSARY,POSE,DISTANCE,EMISSARY,POSE,DISTANCE,EMISSARYPOSE,DISTANCE,EMISSARY,DISTANCE,EMISSARY,DISTANCE,EMISSARY,DISTANCE,EMISSARY,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DISTANCE,DI

In [14]:
import re
from difflib import SequenceMatcher
import numpy as np

def norm_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def tok(s: str):
    return re.findall(r"\w+", s.lower())

def token_f1(pred: str, gold: str) -> float:
    p = tok(pred)
    g = tok(gold)
    if not p or not g:
        return 0.0
    inter = 0
    g_used = [False] * len(g)
    for t in p:
        for i, gt in enumerate(g):
            if not g_used[i] and t == gt:
                inter += 1
                g_used[i] = True
                break
    prec = inter / len(p)
    rec = inter / len(g)
    return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

def rouge_l_f1(pred: str, gold: str) -> float:
    p = tok(pred)
    g = tok(gold)
    if not p or not g:
        return 0.0
    dp = [[0] * (len(g) + 1) for _ in range(len(p) + 1)]
    for i in range(1, len(p) + 1):
        for j in range(1, len(g) + 1):
            if p[i - 1] == g[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    lcs = dp[-1][-1]
    prec = lcs / len(p)
    rec = lcs / len(g)
    return 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)

def exact_match(pred: str, gold: str) -> float:
    return 1.0 if norm_text(pred) == norm_text(gold) else 0.0

def seq_sim(pred: str, gold: str) -> float:
    return SequenceMatcher(None, norm_text(pred), norm_text(gold)).ratio()

scores = {
    "token_f1": [],
    "rouge_l_f1": [],
    "exact_match": [],
    "seq_sim": [],
}

for x in preds:
    scores["token_f1"].append(token_f1(x["prediction"], x["target"]))
    scores["rouge_l_f1"].append(rouge_l_f1(x["prediction"], x["target"]))
    scores["exact_match"].append(exact_match(x["prediction"], x["target"]))
    scores["seq_sim"].append(seq_sim(x["prediction"], x["target"]))

print("Token-F1 medio:", round(float(np.mean(scores["token_f1"])) if scores["token_f1"] else 0.0, 4))
print("ROUGE-L F1 medio:", round(float(np.mean(scores["rouge_l_f1"])) if scores["rouge_l_f1"] else 0.0, 4))
print("Exact-Match medio:", round(float(np.mean(scores["exact_match"])) if scores["exact_match"] else 0.0, 4))
print("SeqSim medio:", round(float(np.mean(scores["seq_sim"])) if scores["seq_sim"] else 0.0, 4))

# Classification metrics (paper-style)
LABELS = ['economical', 'normal', 'aggressive', 'congested']

def extract_label(text):
    if text is None:
        return None
    t = str(text).lower()
    # Prefer explicit Answer: tag
    m = re.findall(r'answer\s*:\s*([a-z_\-]+)', t)
    if m:
        cand = m[-1]
        return cand if cand in LABELS else None
    # Fallback: last label occurrence
    positions = [(t.rfind(lbl), lbl) for lbl in LABELS if t.rfind(lbl) != -1]
    if positions:
        positions.sort()
        return positions[-1][1]
    return None

y_true = []
y_pred = []
for x in preds:
    gt = extract_label(x.get('target'))
    pr = extract_label(x.get('prediction'))
    if gt is None or pr is None:
        continue
    y_true.append(gt)
    y_pred.append(pr)

total = len(y_true)
correct = sum(1 for g,p in zip(y_true, y_pred) if g == p)
accuracy = (correct / total) if total else 0.0

# Macro-F1 over fixed label set
macro_f1 = 0.0
for lbl in LABELS:
    tp = sum(1 for g,p in zip(y_true, y_pred) if g == lbl and p == lbl)
    fp = sum(1 for g,p in zip(y_true, y_pred) if g != lbl and p == lbl)
    fn = sum(1 for g,p in zip(y_true, y_pred) if g == lbl and p != lbl)
    prec = tp / (tp + fp) if (tp + fp) else 0.0
    rec = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 0.0 if (prec + rec) == 0 else 2 * prec * rec / (prec + rec)
    macro_f1 += f1
macro_f1 = macro_f1 / len(LABELS) if LABELS else 0.0

print('Accuracy:', round(float(accuracy), 4), f'({correct}/{total})')
print('Macro-F1:', round(float(macro_f1), 4))


Token-F1 medio: 0.0721
ROUGE-L F1 medio: 0.0357
Exact-Match medio: 0.0
SeqSim medio: 0.0101
Accuracy: 0.6667 (4/6)
Macro-F1: 0.4167


In [15]:
# preds deve existir: lista de dicts com keys: id, prediction, target (e opcionalmente prompt)
# scores deve existir: dict com arrays (token_f1, rouge_l_f1, exact_match, seq_sim)

out_pred = Path("data/obd_eval_predictions.jsonl")
out_summary = Path("data/obd_eval_summary.json")

# salvar previsões
with out_pred.open("w", encoding="utf-8") as f:
    for p in preds:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

# salvar resumo
summary = {
    "n": len(preds),
    "token_f1_mean": float(np.mean(scores["token_f1"])) if scores["token_f1"] else 0.0,
    "rouge_l_f1_mean": float(np.mean(scores["rouge_l_f1"])) if scores["rouge_l_f1"] else 0.0,
    "exact_match_mean": float(np.mean(scores["exact_match"])) if scores["exact_match"] else 0.0,
    "seq_sim_mean": float(np.mean(scores["seq_sim"])) if scores["seq_sim"] else 0.0,
}

out_summary.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

print("Saved predictions to", out_pred)
print("Saved summary to", out_summary)


Saved predictions to data\obd_eval_predictions.jsonl
Saved summary to data\obd_eval_summary.json
